In [1]:
import sys
import os
# 获取当前Notebook的绝对路径
notebook_dir = os.path.abspath('')
# 项目根目录是Notebook所在目录的父目录
project_root = os.path.abspath(os.path.join(notebook_dir, "../../.."))
# 将项目根目录添加到sys.path
if project_root not in sys.path:
    sys.path.insert(0, project_root)
# 确认路径是否正确
print("当前工作目录:", os.getcwd())
print("项目根目录:", project_root)
print("sys.path:", sys.path)
from apt.vendor.tspro.base import base as base
from apt.os.rustfs.RustfsHandler import RustfsClientWrapper

# 创建客户端
rustfs_client = RustfsClientWrapper()

当前工作目录: c:\Users\GHUIQ\repos\MyFunds\jupyterbook\os\Rustfs
项目根目录: c:\Users\GHUIQ\repos\MyFunds
sys.path: ['c:\\Users\\GHUIQ\\repos\\MyFunds', 'c:\\Users\\GHUIQ\\repos\\MyFunds\\.conda\\python311.zip', 'c:\\Users\\GHUIQ\\repos\\MyFunds\\.conda\\DLLs', 'c:\\Users\\GHUIQ\\repos\\MyFunds\\.conda\\Lib', 'c:\\Users\\GHUIQ\\repos\\MyFunds\\.conda', '', 'c:\\Users\\GHUIQ\\repos\\MyFunds\\.conda\\Lib\\site-packages', 'c:\\Users\\GHUIQ\\repos\\MyFunds\\.conda\\Lib\\site-packages\\win32', 'c:\\Users\\GHUIQ\\repos\\MyFunds\\.conda\\Lib\\site-packages\\win32\\lib', 'c:\\Users\\GHUIQ\\repos\\MyFunds\\.conda\\Lib\\site-packages\\Pythonwin']


# RustfsHandler 使用演示

## 更新说明 (2025-07-13)

本演示已适配新的环境变量配置：

- **环境变量名称**: 使用英文大写格式 `RUSTFS_ENDPOINT`、`RUSTFS_ACCESS_KEY`、`RUSTFS_SECRET_KEY`、`RUSTFS_CACHE_PATH`
- **配置文件位置**: 配置信息存储在项目根目录的 `.env` 文件中
- **当前配置**:
  - RUSTFS_ENDPOINT = "192.168.1.201:29000"
  - RUSTFS_ACCESS_KEY = "rustfsadmin"
  - RUSTFS_SECRET_KEY = "rustfsadmin"
  - RUSTFS_CACHE_PATH = "c:\minio_cache"

## 技术架构

- **协议**: 使用 S3 兼容协议通过 boto3 SDK 连接
- **服务**: RustFS 对象存储服务
- **客户端**: RustfsClientWrapper 封装类

## 功能演示

本 Notebook 演示了 RustfsClientWrapper 的主要功能：

1. **初始化和连接** - 使用环境变量自动配置
2. **桶管理** - 列出、创建、删除桶
3. **文件管理** - 上传、下载、列出文件
4. **配置验证** - 验证环境变量和连接状态

In [2]:
# 列出所有的桶及其信息
rustfs_client.list_buckets()

,桶名称,创建日期,总文件数,总大小
0,hdf5,2025-07-13 03:29,0,0.0 B
1,test,2025-07-13 03:30,7,26.3 GB


In [4]:
# 新建和删除桶
bucket_name = "test"
rustfs_client.create_bucket(bucket_name)
print("创建桶后:")
print(rustfs_client.list_buckets())

rustfs_client.delete_bucket(bucket_name)
print("删除桶后:")
print(rustfs_client.list_buckets())

创建桶后:
    桶名称              创建日期  总文件数    总大小
0  hdf5  2025-07-13 03:29     0  0.0 B
1  test  2025-07-13 13:59     0  0.0 B
删除桶后:
    桶名称              创建日期  总文件数    总大小
0  hdf5  2025-07-13 03:29     0  0.0 B


In [5]:
# 列出桶的文件列表
bucket_name = "hdf5"
# 列出某一个或多个文件 使用prefix做过滤
#rustfs_client.list_files(bucket_name,prefix="akshare/data/1min/000001.SZ.h5")
# 列出全部文件
rustfs_client.list_files(bucket_name)

""


In [ ]:
# 上传文件
import tkinter as tk
from tkinter import filedialog
bucket_name = "share"
# 创建一个隐藏的Tkinter窗口
root = tk.Tk()
root.withdraw()
# 弹出文件选择对话框
file_path = filedialog.askopenfilename()
file_name = os.path.basename(file_path)
# 上传文件到Rustfs
if file_path:
    rustfs_client.upload_file(bucket_name, file_name, file_path)
    print(f"文件 {file_path} 已上传到桶 {bucket_name}")
else:
    print("未选择文件")

文件 C:/Users/george/Documents/乔治U盘/qzx3(2)/qzx(1)/(1)/PPT/青花瓷（2024.4.25）(小报).pptx 已上传到桶 share


In [6]:
# 验证环境变量配置
import os
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()

print("=== Rustfs 环境变量配置验证 ===")
print(f"RUSTFS_ENDPOINT: {os.getenv('RUSTFS_ENDPOINT')}")
print(f"RUSTFS_ACCESS_KEY: {os.getenv('RUSTFS_ACCESS_KEY')}")
print(f"RUSTFS_SECRET_KEY: {os.getenv('RUSTFS_SECRET_KEY')}")
print(f"RUSTFS_CACHE_PATH: {os.getenv('RUSTFS_CACHE_PATH')}")

print("\n=== RustfsClientWrapper 配置验证 ===")
print(f"客户端缓存路径: {rustfs_client.RUSTFS_CACHE_PATH}")
print(f"缓存目录是否存在: {os.path.exists(rustfs_client.RUSTFS_CACHE_PATH)}")

# 验证连接
try:
    buckets = rustfs_client.list_buckets()
    print(f"✓ 连接成功，发现 {len(buckets)} 个存储桶")
except Exception as e:
    print(f"✗ 连接失败: {e}")

=== Rustfs 环境变量配置验证 ===
RUSTFS_ENDPOINT: 192.168.1.201:29000
RUSTFS_ACCESS_KEY: rustfsadmin
RUSTFS_SECRET_KEY: rustfsadmin
RUSTFS_CACHE_PATH: c:\minio_cache

=== RustfsClientWrapper 配置验证 ===
客户端缓存路径: c:\minio_cache
缓存目录是否存在: True
✓ 连接成功，发现 1 个存储桶


In [ ]:
# 文件下载功能测试
import os

# 测试文件下载
bucket_name = "hdf5"
object_name = "akshare/data/1min/000001.SZ.h5"  # 选择一个存在的文件
local_file_name = "test_download.h5"
download_path = os.path.join(rustfs_client.RUSTFS_CACHE_PATH, local_file_name)

print(f"正在下载文件...")
print(f"桶名称: {bucket_name}")
print(f"对象名称: {object_name}")
print(f"本地路径: {download_path}")

# 检查文件是否存在
if rustfs_client.file_exists(bucket_name, object_name):
    print("✓ 文件存在于 Rustfs 中")
    
    # 下载文件
    success = rustfs_client.download_file(bucket_name, object_name, download_path)
    
    if success:
        print("✓ 文件下载成功")
        print(f"文件大小: {os.path.getsize(download_path)} 字节")
        
        # 清理测试文件
        if os.path.exists(download_path):
            os.remove(download_path)
            print("✓ 测试文件已清理")
    else:
        print("✗ 文件下载失败")
else:
    print("✗ 文件不存在于 Rustfs 中")

正在下载文件...
桶名称: hdf5
对象名称: akshare/data/1min/000001.SZ.h5
本地路径: c:\minio_cache\test_download.h5
✗ 文件不存在于 Rustfs 中


: 